In [1]:
import torch
import testdata
from _em import _EM_step_full_stable, fit_EM_iter

In [2]:
%load_ext autoreload

In [3]:
%autoreload 2

Generate fake data

In [4]:
params = {
    'd': 10, 
    'k': [5, 3, 4], 
    'p': [15, 13, 14], 
    'n': 5000,
    'sigsq': [0.3, 0.7, 0.5]
}

Complete data case

In [ ]:
def run_test(params, n_iter=1000, data='complete'):
    metrics = {
        'WWt_corr': [],
        'LLt_corr': [],
        'Phi_corr': [],
        'Sigma_corr': [] # WW^T + LL^T + Phi
    }
    
    for i in range(n_iter):
        if (i+1) % (0.2*n_iter) == 0:
            print(f"{i+1} simulations completed")

        # generate data
        Y, W_true, L_true, Phi_true = testdata.simulate_data(params, private_var=True)
        Y = Y.T
    
        if data=='missing_disjoint':
            Y[1000:1025, :params['p'][0]] = float('nan')
            Y[1025:1075, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
            Y[1075:1100, params['p'][0]+params['p'][1]:] = float('nan')
            
        elif data=='missing_overlap':
            Y[1000:1050, :params['p'][0]] = float('nan')
            Y[1040:1100, params['p'][0]:params['p'][0]+params['p'][1]] = float('nan')
            Y[1075:1150, params['p'][0]+params['p'][1]:] = float('nan')

        W_init, L_init, Phi_init = testdata.initialize_params(W_true, L_true, Phi_true, private_var=True)
        Sigma_hat = Y.nan_to_num().T @ Y.nan_to_num() / Y.shape[0]

        # ground truths
        WW = (torch.cat(W_true, dim=0)) @ (torch.cat(W_true, dim=0).T)
        LL = (torch.block_diag(*L_true)) @ (torch.block_diag(*L_true).T)
        P = torch.cat([Phi.flatten() for Phi in Phi_true])
        Sigma_true = torch.cat([
            torch.flatten(
                (W_true[i] @ W_true[i].T) + 
                (L_true[i] @ L_true[i].T) + 
                Phi_true[i]
            ) 
            for i in range(len(W_true))
        ])
    
        # run EM
        W_new, L_new, Phi_new, _, _ = fit_EM_iter(
            Y, Sigma_hat, W_init, L_init, Phi_init, maxit=1000, impute=(data!='complete')
        )
        WW_test = (torch.cat(W_new, dim=0)) @ (torch.cat(W_new, dim=0).T)
        LL_test = (torch.block_diag(*L_new)) @ (torch.block_diag(*L_new).T)
        P_test = torch.cat([Phi.flatten() for Phi in Phi_new])
        Sigma_test = torch.cat([
            torch.flatten(
                (W_new[i] @ W_new[i].T) + 
                (L_new[i] @ L_new[i].T) +
                Phi_new[i]
            ) 
            for i in range(len(W_new))
        ])
        
        # get metrics
        metrics['WWt_corr'].append(torch.corrcoef(
            torch.stack([WW.flatten(), WW_test.flatten()], dim=0)
        )[0,1].item())
        metrics['LLt_corr'].append(torch.corrcoef(
            torch.stack([LL.flatten(), LL_test.flatten()], dim=0)
        )[0,1].item())
        metrics['Phi_corr'].append(torch.corrcoef(
            torch.stack([P, P_test], dim=0)
        )[0,1].item())
        metrics['Sigma_corr'].append(torch.corrcoef(
            torch.stack([Sigma_true, Sigma_test], dim=0)
        )[0,1].item())

    print('\nFinal Metrics:')
    for k, v in metrics.items():
        print(' ', k)
        v = torch.tensor(v)
        print(f"\tMean: {round(torch.mean(v).item(), 4)}")
        print(f"\tMin: {round(torch.min(v).item(), 4)}")
        print(f"\tMax: {round(torch.max(v).item(), 4)}")

Complete data

In [6]:
run_test(params)

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed

Metrics:
  WWt_corr
	Mean: 0.9973
	Min: 0.9728
	Max: 0.9987
  LLt_corr
	Mean: 0.9993
	Min: 0.9026
	Max: 0.9998
  Phi_corr
	Mean: 0.9984
	Min: 0.803
	Max: 0.9996
  Sigma_corr
	Mean: 0.9986
	Min: 0.9968
	Max: 0.9993


Missing data, with each sample missing maximum 1 mode

In [7]:
run_test(params, data='missing_disjoint')

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed

Metrics:
  WWt_corr
	Mean: 0.9974
	Min: 0.9948
	Max: 0.9986
  LLt_corr
	Mean: 0.9993
	Min: 0.9979
	Max: 0.9997
  Phi_corr
	Mean: 0.9946
	Min: 0.9636
	Max: 0.9985
  Sigma_corr
	Mean: 0.9987
	Min: 0.997
	Max: 0.9994


Missing data, with each sample missing up to 2/3 modes

In [8]:
run_test(params, data='missing_overlap')

200 simulations completed
400 simulations completed
600 simulations completed
800 simulations completed
1000 simulations completed

Metrics:
  WWt_corr
	Mean: 0.9974
	Min: 0.9943
	Max: 0.9987
  LLt_corr
	Mean: 0.9992
	Min: 0.9975
	Max: 0.9997
  Phi_corr
	Mean: 0.9877
	Min: 0.9272
	Max: 0.9969
  Sigma_corr
	Mean: 0.9987
	Min: 0.9963
	Max: 0.9993
